In [0]:
# 1. Load the data from your Bronze table
df_bronze = spark.table("workspace.ckd_project.ckd_bronze")

# 2. Print the Schema (The "Skeleton" of your data)
print(" Current Data Types (Bronze) ")
df_bronze.printSchema()

# 3. View the first 5 rows to see the raw text
print("Raw Data Samples")
display(df_bronze.limit(5))

 Current Data Types (Bronze) 
root
 |-- bp_diastolic: string (nullable = true)
 |-- bp_limit: string (nullable = true)
 |-- sg: string (nullable = true)
 |-- al: string (nullable = true)
 |-- class: string (nullable = true)
 |-- rbc: string (nullable = true)
 |-- su: string (nullable = true)
 |-- pc: string (nullable = true)
 |-- pcc: string (nullable = true)
 |-- ba: string (nullable = true)
 |-- bgr: string (nullable = true)
 |-- bu: string (nullable = true)
 |-- sod: string (nullable = true)
 |-- sc: string (nullable = true)
 |-- pot: string (nullable = true)
 |-- hemo: string (nullable = true)
 |-- pcv: string (nullable = true)
 |-- rbcc: string (nullable = true)
 |-- wbcc: string (nullable = true)
 |-- htn: string (nullable = true)
 |-- dm: string (nullable = true)
 |-- cad: string (nullable = true)
 |-- appet: string (nullable = true)
 |-- pe: string (nullable = true)
 |-- ane: string (nullable = true)
 |-- grf: string (nullable = true)
 |-- stage: string (nullable = true)
 |-- a

bp_diastolic,bp_limit,sg,al,class,rbc,su,pc,pcc,ba,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,0,1.019 - 1.021,1 - 1,ckd,0,< 0,0,0,0,< 112,< 48.1,138 - 143,< 3.65,< 7.31,11.3 - 12.6,33.5 - 37.4,4.46 - 5.05,7360 - 9740,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
0,0,1.009 - 1.011,< 0,ckd,0,< 0,0,0,0,112 - 154,< 48.1,133 - 138,< 3.65,< 7.31,11.3 - 12.6,33.5 - 37.4,4.46 - 5.05,12120 - 14500,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
0,0,1.009 - 1.011,≥ 4,ckd,1,< 0,1,0,1,< 112,48.1 - 86.2,133 - 138,< 3.65,< 7.31,8.7 - 10,29.6 - 33.5,4.46 - 5.05,14500 - 16880,0,0,0,1,0,0,127.281 - 152.446,s1,1,< 12
1,1,1.009 - 1.011,3 - 3,ckd,0,< 0,0,0,0,112 - 154,< 48.1,133 - 138,< 3.65,< 7.31,13.9 - 15.2,41.3 - 45.2,4.46 - 5.05,7360 - 9740,0,0,0,0,0,0,127.281 - 152.446,s1,1,< 12
0,0,1.015 - 1.017,< 0,ckd,0,< 0,0,0,0,154 - 196,< 48.1,133 - 138,< 3.65,< 7.31,13.9 - 15.2,37.4 - 41.3,5.05 - 5.64,7360 - 9740,0,1,0,1,1,0,127.281 - 152.446,s1,1,12 - 20


In [0]:
from pyspark.sql.functions import col, trim, split

df = spark.table("workspace.ckd_project.ckd_bronze")

In [0]:
from pyspark.sql.functions import col, when, trim, regexp_replace, lower, split, expr

df_bronze = spark.table("workspace.ckd_project.ckd_bronze")

df_clean = df_bronze.select([
    when(trim(col(c)).isin(["?", ""]), None)
    .otherwise(trim(col(c)))
    .alias(c) 
    for c in df_bronze.columns
])

numeric_cols = ["sg", "al", "bgr", "bu", "sod", "sc", "pot", "hemo", 
                "pcv", "rbcc", "wbcc", "grf", "bp_diastolic", "age",
                "su", "bp_limit", "affected"]

for col_name in numeric_cols:
    df_clean = df_clean.withColumn(
        col_name,
        when(col(col_name).isNull(), None)
        .otherwise(
            when(
                col(col_name).rlike("\\d+\\.?\\d*\\s*-\\s*\\d+\\.?\\d*"),
                expr(f"cast((cast(split({col_name}, '\\\\s*-\\\\s*')[0] as double) + cast(split({col_name}, '\\\\s*-\\\\s*')[1] as double))/2 as string)")
            ).otherwise(
                regexp_replace(
                    regexp_replace(col(col_name), "^[<≥>=]+\\s*", ""),
                    "\\s*-\\s*.*$", ""
                )
            )
        )
    )

text_cols = ["class", "rbc", "pc", "pcc", "ba", "htn", "dm", 
             "cad", "appet", "pe", "ane", "stage"]

for col_name in text_cols:
    df_clean = df_clean.withColumn(col_name, lower(col(col_name)))

print(" Data Cleaning Complete!")
print(f"\nRows: {df_clean.count()}")
print("\n Sample cleaned data:")
display(df_clean.limit(10))

✅ Data Cleaning Complete!

Rows: 200

📊 Sample cleaned data:


bp_diastolic,bp_limit,sg,al,class,rbc,su,pc,pcc,ba,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,0,1.02,1.0,ckd,0,0,0,0,0,112,48.1,140.5,3.65,7.31,11.95,35.45,4.755,8550.0,0,0,0,0,0,0,227.944,s1,1,12
0,0,1.0099999999999998,0,ckd,0,0,0,0,0,133.0,48.1,135.5,3.65,7.31,11.95,35.45,4.755,13310.0,0,0,0,0,0,0,227.944,s1,1,12
0,0,1.0099999999999998,4,ckd,1,0,1,0,1,112,67.15,135.5,3.65,7.31,9.35,31.55,4.755,15690.0,0,0,0,1,0,0,139.8635,s1,1,12
1,1,1.0099999999999998,3.0,ckd,0,0,0,0,0,133.0,48.1,135.5,3.65,7.31,14.55,43.25,4.755,8550.0,0,0,0,0,0,0,139.8635,s1,1,12
0,0,1.016,0,ckd,0,0,0,0,0,175.0,48.1,135.5,3.65,7.31,14.55,39.349999999999994,5.345,8550.0,0,1,0,1,1,0,139.8635,s1,1,16.0
1,1,1.023,0,notckd,0,0,0,0,0,112,48.1,135.5,3.65,7.31,16.5,49.1,5.345,6170.0,0,0,0,0,0,0,114.69800000000001,s1,0,16.0
0,0,1.02,3.0,ckd,0,0,0,0,0,112,48.1,140.5,3.65,7.31,10.65,31.55,3.575,8550.0,1,1,0,0,0,0,190.195,s1,1,16.0
0,0,1.02,0,ckd,0,0,0,0,0,133.0,67.15,135.5,3.65,7.31,11.95,39.349999999999994,4.755,6170.0,0,0,0,0,0,0,39.20035,s4,1,16.0
0,0,1.023,0,notckd,0,0,0,0,0,133.0,67.15,135.5,3.65,7.31,14.55,39.349999999999994,5.345,4980,0,0,0,0,0,0,39.20035,s4,0,23.5
1,2,1.0099999999999998,4,ckd,0,0,1,1,1,112,48.1,125.5,3.65,7.31,8.05,23.75,4.165,13310.0,0,0,0,0,0,1,64.3661,s3,1,23.5


In [0]:
from pyspark.sql.functions import col, when

# Define column types
double_cols = ["sg", "bgr", "bu", "sod", "sc", "pot", "hemo", "pcv", "rbcc", "wbcc", "grf"]
int_cols = ["age", "al", "su", "bp_diastolic", "bp_limit", "affected"]

df_silver = df_clean
for col_name in double_cols:
    df_silver = df_silver.withColumn(
        col_name,
        when(col(col_name).rlike("^-?\\d+\\.?\\d*$"), col(col_name).cast("double"))
        .otherwise(None)
    )
for col_name in int_cols:
    df_silver = df_silver.withColumn(
        col_name,
        when(col(col_name).rlike("^-?\\d+\\.?\\d*$"), col(col_name).cast("double").cast("int"))
        .otherwise(None)
    )
df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.ckd_project.ckd_silver")

print("✅ Silver table created successfully!")
print("\n📊 Final Schema:")
df_silver.printSchema()
print("\n Sample Silver data:")
display(spark.table("workspace.ckd_project.ckd_silver").limit(10))

✅ Silver table created successfully!

📊 Final Schema:
root
 |-- bp_diastolic: integer (nullable = true)
 |-- bp_limit: integer (nullable = true)
 |-- sg: double (nullable = true)
 |-- al: integer (nullable = true)
 |-- class: string (nullable = true)
 |-- rbc: string (nullable = true)
 |-- su: integer (nullable = true)
 |-- pc: string (nullable = true)
 |-- pcc: string (nullable = true)
 |-- ba: string (nullable = true)
 |-- bgr: double (nullable = true)
 |-- bu: double (nullable = true)
 |-- sod: double (nullable = true)
 |-- sc: double (nullable = true)
 |-- pot: double (nullable = true)
 |-- hemo: double (nullable = true)
 |-- pcv: double (nullable = true)
 |-- rbcc: double (nullable = true)
 |-- wbcc: double (nullable = true)
 |-- htn: string (nullable = true)
 |-- dm: string (nullable = true)
 |-- cad: string (nullable = true)
 |-- appet: string (nullable = true)
 |-- pe: string (nullable = true)
 |-- ane: string (nullable = true)
 |-- grf: double (nullable = true)
 |-- stage: str

bp_diastolic,bp_limit,sg,al,class,rbc,su,pc,pcc,ba,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,0,1.02,1,ckd,0,0,0,0,0,112.0,48.1,140.5,3.65,7.31,11.95,35.45,4.755,8550.0,0,0,0,0,0,0,227.944,s1,1,12
0,0,1.0099999999999998,0,ckd,0,0,0,0,0,133.0,48.1,135.5,3.65,7.31,11.95,35.45,4.755,13310.0,0,0,0,0,0,0,227.944,s1,1,12
0,0,1.0099999999999998,4,ckd,1,0,1,0,1,112.0,67.15,135.5,3.65,7.31,9.35,31.55,4.755,15690.0,0,0,0,1,0,0,139.8635,s1,1,12
1,1,1.0099999999999998,3,ckd,0,0,0,0,0,133.0,48.1,135.5,3.65,7.31,14.55,43.25,4.755,8550.0,0,0,0,0,0,0,139.8635,s1,1,12
0,0,1.016,0,ckd,0,0,0,0,0,175.0,48.1,135.5,3.65,7.31,14.55,39.349999999999994,5.345,8550.0,0,1,0,1,1,0,139.8635,s1,1,16
1,1,1.023,0,notckd,0,0,0,0,0,112.0,48.1,135.5,3.65,7.31,16.5,49.1,5.345,6170.0,0,0,0,0,0,0,114.69800000000001,s1,0,16
0,0,1.02,3,ckd,0,0,0,0,0,112.0,48.1,140.5,3.65,7.31,10.65,31.55,3.575,8550.0,1,1,0,0,0,0,190.195,s1,1,16
0,0,1.02,0,ckd,0,0,0,0,0,133.0,67.15,135.5,3.65,7.31,11.95,39.349999999999994,4.755,6170.0,0,0,0,0,0,0,39.20035,s4,1,16
0,0,1.023,0,notckd,0,0,0,0,0,133.0,67.15,135.5,3.65,7.31,14.55,39.349999999999994,5.345,4980.0,0,0,0,0,0,0,39.20035,s4,0,23
1,2,1.0099999999999998,4,ckd,0,0,1,1,1,112.0,48.1,125.5,3.65,7.31,8.05,23.75,4.165,13310.0,0,0,0,0,0,1,64.3661,s3,1,23
